In [1]:
# ===== BOOTSTRAP: run first after every Colab start/restart =====
import os, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

# 1) Reconnect Google Drive (permanent storage)
drive.mount('/content/drive')

# 2) Load secrets from the Colab Secrets panel into environment variables
SECRETS = ["GITHUB_USERNAME", "GITHUB_TOKEN", "GIT_EMAIL", "DAGSHUB_USERNAME",
           "DAGSHUB_TOKEN", "KAGGLE_API_TOKEN"]
missing = []
for k in SECRETS:
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        missing.append(k)
print(f"Secrets loaded: {len(SECRETS)-len(missing)}/{len(SECRETS)}  missing: {missing or 'none'}")

# 3) Project paths
PROJECT     = "23L-0951_A2_Ops"
DRIVE       = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE / PROJECT              # git repo (code, notebooks, small files)
WORK_DIR    = DRIVE / f"{PROJECT}_work"    # big files kept OUT of git
LOCAL_DATA  = Path("/content/data")        # fast local copy, rebuilt each session
for p in (WORK_DIR, LOCAL_DATA):
    p.mkdir(parents=True, exist_ok=True)
os.environ.update(PROJECT_DIR=str(PROJECT_DIR), WORK_DIR=str(WORK_DIR),
                  LOCAL_DATA=str(LOCAL_DATA))
os.environ["DVC_NO_ANALYTICS"] = "1"

def sh(cmd, cwd=None):
    """Run a shell command and print its output. Returns the exit code."""
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    return r.returncode

# 4) Git settings: one credential helper per website, tokens read from env vars.
#    GitHub gets the GitHub token, DagsHub gets the DagsHub token.
#    Tokens are never saved inside the repo or in remote URLs.
gh_helper = '!f() { echo "username=${GITHUB_USERNAME}"; echo "password=${GITHUB_TOKEN}"; }; f'
dh_helper = '!f() { echo "username=${DAGSHUB_USERNAME}"; echo "password=${DAGSHUB_TOKEN}"; }; f'
subprocess.run(["git", "config", "--global", "--unset-all", "credential.helper"])  # remove old generic helper
for key, val in [("credential.https://github.com.helper", gh_helper),
                 ("credential.https://dagshub.com.helper", dh_helper),
                 ("user.name", "Zayan"),
                 ("user.email", os.environ.get("GIT_EMAIL", "")),
                 ("init.defaultBranch", "main")]:
    subprocess.run(["git", "config", "--global", key, val], check=True)

# 5) Clone the repo into Drive the first time; afterwards just pull updates
if not (PROJECT_DIR / ".git").exists():
    sh(f'git clone https://github.com/{os.environ["GITHUB_USERNAME"]}/{PROJECT}.git "{PROJECT_DIR}"')
if not (PROJECT_DIR / ".git").exists():
    raise SystemExit("Clone failed: check GITHUB_USERNAME / GITHUB_TOKEN / repo name.")
os.chdir(PROJECT_DIR)
sh("rm -f .git/index.lock")      # clears a leftover lock from a dropped session
sh("git pull --ff-only")

# 6) Reinstall pinned packages
if (PROJECT_DIR / "requirements-colab.txt").exists():
    sh("pip install -q -r requirements-colab.txt")

# 7) Helper to save a step: copy notebook into repo, commit, push
def save_step(message, notebook_name=None):
    if notebook_name:
        src = DRIVE / "Colab Notebooks" / notebook_name
        (PROJECT_DIR / "notebooks").mkdir(exist_ok=True)
        if src.exists():
            shutil.copy(src, PROJECT_DIR / "notebooks" / notebook_name)
        else:
            print("Notebook not found:", src)
    sh("git add -A")
    sh(f'git commit -m "{message}"')
    sh("git push origin main")
    sh("git log --oneline -1")

print("Bootstrap done. Working directory:", os.getcwd())

Mounted at /content/drive
Secrets loaded: 6/6  missing: none
Already up to date.
Bootstrap done. Working directory: /content/drive/MyDrive/23L-0951_A2_Ops


In [2]:
import zipfile, time, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf, keras, mlflow

# Fast local copy of the images (rebuilt from the zip on Drive each session)
RAW_LOCAL = LOCAL_DATA / "raw"
ZIP_PATH = next((WORK_DIR / "raw_zip").glob("*.zip"))
if not (RAW_LOCAL.exists() and any(RAW_LOCAL.rglob("*.*"))):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(RAW_LOCAL)
print("Local images:", sum(1 for p in RAW_LOCAL.rglob("*") if p.is_file()))

# MLflow -> DagsHub, credentials from Colab Secrets via environment variables
os.environ["MLFLOW_TRACKING_URI"]      = f"https://dagshub.com/{os.environ['DAGSHUB_USERNAME']}/{PROJECT}.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = os.environ["DAGSHUB_USERNAME"]
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.environ["DAGSHUB_TOKEN"]
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

# Folders for checkpoints/tuner results on Drive (survive disconnects)
CKPT_DIR  = WORK_DIR / "checkpoints"; CKPT_DIR.mkdir(exist_ok=True)
TUNER_DIR = WORK_DIR / "tuner";       TUNER_DIR.mkdir(exist_ok=True)
FIG_DIR   = PROJECT_DIR / "reports" / "figures"

SEED = 42
keras.utils.set_random_seed(SEED)       # seeds Python, NumPy and TensorFlow together
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

Local images: 975
TF 2.21.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
%%writefile src/data.py
"""
VisionGuard input pipeline (Task 2.1).

decode_and_resize() is shared by training and the Flask app, so both
apply exactly the same preprocessing. Models receive pixels in 0..255
and normalise internally, so serving cannot use the wrong scaling.
"""
from pathlib import Path

import pandas as pd
import tensorflow as tf
import keras

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
CLASS_NAMES = {0: "real", 1: "ai"}

AUG_SETTINGS = {
    "mild":   dict(rotation=0.05, zoom=0.10, contrast=0.10),
    "strong": dict(rotation=0.15, zoom=0.25, contrast=0.25),
}


def decode_and_resize(raw_bytes, img_size=IMG_SIZE):
    """Encoded image bytes -> uint8 tensor (img_size, img_size, 3).

    - decode_image handles JPEG/PNG/GIF/WEBP/BMP; channels=3 converts
      grayscale and drops alpha; expand_animations=False takes a GIF's first frame.
    - resize_with_pad keeps the aspect ratio and fills the rest with black
      (letterboxing) instead of stretching the image.
    """
    img = tf.io.decode_image(raw_bytes, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize_with_pad(img, img_size, img_size, antialias=True)
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)


def load_image(path, img_size=IMG_SIZE):
    return decode_and_resize(tf.io.read_file(path), img_size)


def make_augmenter(strength="mild", seed=42):
    """Random augmentation for TRAINING only. Returns None for 'none'."""
    if strength == "none":
        return None
    s = AUG_SETTINGS[strength]
    return keras.Sequential([
        keras.layers.RandomFlip("horizontal", seed=seed),
        keras.layers.RandomRotation(s["rotation"], fill_mode="constant", seed=seed),
        keras.layers.RandomZoom(s["zoom"], fill_mode="constant", seed=seed),
        keras.layers.RandomContrast(s["contrast"], seed=seed),
    ], name=f"augment_{strength}")


def read_split(split_csv, data_root):
    df = pd.read_csv(split_csv)
    paths = [str(Path(data_root) / p) for p in df["rel_path"]]
    return paths, df["label"].astype("int32").tolist(), df


def make_dataset(split_csv, data_root, training, img_size=IMG_SIZE,
                 batch_size=BATCH_SIZE, augment="mild", seed=42):
    """Split list -> batched, prefetched tf.data.Dataset of (images 0..255 float32, labels)."""
    paths, labels, _ = read_split(split_csv, data_root)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    # 1) read + decode + resize, in parallel
    ds = ds.map(lambda p, y: (load_image(p, img_size), y), num_parallel_calls=AUTOTUNE)
    # 2) cache decoded uint8 images in RAM: decoding happens only once
    ds = ds.cache()
    # 3) shuffle training data only; new order every epoch
    if training:
        ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)
    # 4) batch, then convert to float for the model
    ds = ds.batch(batch_size)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), tf.cast(y, tf.float32)),
                num_parallel_calls=AUTOTUNE)
    # 5) augmentation on training batches only (never on val/test, never inside the saved model)
    aug = make_augmenter(augment, seed) if training else None
    if aug is not None:
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
    # 6) prefetch: prepare the next batch while the GPU trains on the current one
    return ds.prefetch(AUTOTUNE)

Writing src/data.py


In [ ]:
import importlib, sys
sys.path.insert(0, str(PROJECT_DIR))
import src.data as data; importlib.reload(data)

SPLITS = PROJECT_DIR / "data" / "splits"
train_ds = data.make_dataset(SPLITS / "train.csv", RAW_LOCAL, training=True)
val_ds   = data.make_dataset(SPLITS / "val.csv",   RAW_LOCAL, training=False)
test_ds  = data.make_dataset(SPLITS / "test.csv",  RAW_LOCAL, training=False)

print("Element spec:", train_ds.element_spec)
for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    print(f"{name:5s}: {sum(1 for _ in ds)} batches")

# Show the effect of caching: the 1st pass decodes from disk, the 2nd reads from RAM
for i in (1, 2):
    t = time.time(); n = sum(int(x.shape[0]) for x, _ in train_ds)
    print(f"Pass {i}: {n} images in {time.time()-t:.1f}s")

Element spec: (TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.float32, name=None))
train: 22 batches


In [ ]:
# Figure 1: one augmented training batch
images, labels = next(iter(train_ds))
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, img, y in zip(axes.flat, images[:16], labels[:16]):
    ax.imshow(np.clip(img.numpy() / 255.0, 0, 1))
    ax.set_title(data.CLASS_NAMES[int(y)], fontsize=11)
    ax.axis("off")
fig.suptitle("One augmented training batch (224×224, letterboxed, mild augmentation)", fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "pipeline_augmented_batch.png", dpi=110, bbox_inches="tight")
plt.show()

# Figure 2: what happens to aspect ratio (original vs stretch vs letterbox)
from PIL import Image
audit = pd.read_csv("reports/audit/audit_table.csv")
wide = audit[audit.readable].assign(ar=lambda d: d.width / d.height).sort_values("ar").iloc[-5]
raw = tf.io.read_file(str(RAW_LOCAL / wide.rel_path))
orig = tf.io.decode_image(raw, channels=3, expand_animations=False)
stretched = tf.cast(tf.image.resize(orig, (224, 224)), tf.uint8)
padded = data.decode_and_resize(raw)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, im, title in zip(axes, [orig, stretched, padded],
                         [f"Original {int(wide.width)}×{int(wide.height)}",
                          "Plain resize 224×224 (stretched)",
                          "resize_with_pad 224×224 (used)"]):
    ax.imshow(im.numpy()); ax.set_title(title); ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "pipeline_aspect_ratio.png", dpi=110, bbox_inches="tight")
plt.show()